# Question 2 — Mumbai 2020 tropospheric NO₂ and residents aged 65+

**Aditi Saha · HS25M101 · Master of Public Policy, IIT Tirupati**  
**Course:** Urban Informatics and Spatial Analytics

## Assigned task
Map **2020 mean tropospheric NO₂** for Greater Mumbai. Estimate the number and share of residents **aged 65+** living in the city's **highest-NO₂ quartile**.

## Datasets
- Sentinel-5P OFFL L3 NO₂: `COPERNICUS/S5P/OFFL/L3_NO2` (2020)
- WorldPop age–sex: `WorldPop/GP/100m/pop_age_sex_cons_unadj` (India, 2020)

## Required outputs
1. Monthly 2020 NO₂ GIF / time slider  
2. Annual mean NO₂ map  
3. 65+ exposure map and summary table  
4. Interpretation  

## City boundary
Greater Mumbai ULB from `dataset/Mumbai_boundary.geojson`  
(source: Indian urban ULB release / SBM ULBs via [ramSeraph/indian_admin_boundaries urban release](https://github.com/ramSeraph/indian_admin_boundaries/releases/tag/urban)).


In [ ]:
# 1. Import libraries
from pathlib import Path
import json

import ee
import geemap
import geopandas as gpd
import pandas as pd
import requests
from PIL import Image, ImageDraw

PROJECT_ID = "solar-grail-510708-p8"
print("Project ID:", PROJECT_ID)


In [ ]:
# 2. Initialize Google Earth Engine
# If this fails, run once: ee.Authenticate()
ee.Initialize(project=PROJECT_ID)
print("Earth Engine initialized successfully.")

message = ee.String("Connected to Google Earth Engine")
print(message.getInfo())


In [ ]:
# 3. Paths (repo root or question2/ as working directory)
if Path("../dataset/Mumbai_boundary.geojson").exists():
    BOUNDARY_PATH = Path("../dataset/Mumbai_boundary.geojson")
    OUT_DIR = Path("../web/q2/outputs")
elif Path("dataset/Mumbai_boundary.geojson").exists():
    BOUNDARY_PATH = Path("dataset/Mumbai_boundary.geojson")
    OUT_DIR = Path("web/q2/outputs")
else:
    BOUNDARY_PATH = Path(r"C:/Users/Ā/Documents/python_class/dataset/Mumbai_boundary.geojson")
    OUT_DIR = Path(r"C:/Users/Ā/Documents/python_class/web/q2/outputs")

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Boundary:", BOUNDARY_PATH.resolve())
print("Outputs:", OUT_DIR.resolve())


In [ ]:
# 4. Load Greater Mumbai boundary and convert to Earth Engine
mumbai_gdf = gpd.read_file(BOUNDARY_PATH)

if mumbai_gdf.crs is None or mumbai_gdf.crs.to_epsg() != 4326:
    mumbai_gdf = mumbai_gdf.to_crs(epsg=4326)

print(mumbai_gdf[["ulbnm", "ulbcode", "distnm", "stnm"]])
print("CRS:", mumbai_gdf.crs)
print("Bounds:", mumbai_gdf.total_bounds)

mumbai_ee = geemap.geopandas_to_ee(mumbai_gdf)
print("Mumbai boundary converted to Earth Engine.")


In [ ]:
# 5. Preview city boundary
Map = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map


## 6. Sentinel-5P tropospheric NO₂ (2020)

Band: `tropospheric_NO2_column_number_density` (mol/m²).  
Pixels with `cloud_fraction > 0.2` are masked before averaging.  
Annual and monthly maps use the **same colour scale**.


In [ ]:
# 6. Load and filter Sentinel-5P NO2 for Mumbai, 2020
NO2_BAND = "tropospheric_NO2_column_number_density"
CLOUD_BAND = "cloud_fraction"

no2 = ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")

no2_2020 = (
    no2
    .filterBounds(mumbai_ee.geometry())
    .filterDate("2020-01-01", "2021-01-01")
)

print("Number of NO2 images:", no2_2020.size().getInfo())

# Shared visualisation scale for annual + monthly maps
no2_vis = {
    "min": 0,
    "max": 0.0002,
    "palette": ["blue", "cyan", "yellow", "orange", "red"],
}


def mask_clouds(img):
    return img.updateMask(img.select(CLOUD_BAND).lte(0.2))


In [ ]:
# 7. Annual mean tropospheric NO2 for 2020
annual_no2 = (
    no2_2020
    .map(mask_clouds)
    .select(NO2_BAND)
    .mean()
    .clip(mumbai_ee.geometry())
)

Map_annual = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map_annual.addLayer(annual_no2, no2_vis, "Annual Mean NO2 - 2020")
Map_annual.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map_annual.add_colorbar(no2_vis, label="Tropospheric NO2 (mol/m2)")
Map_annual


In [ ]:
# 8. Export annual NO2 map for the webpage
region = mumbai_ee.geometry().bounds()

thumb_annual = annual_no2.getThumbURL({
    "min": no2_vis["min"],
    "max": no2_vis["max"],
    "palette": no2_vis["palette"],
    "region": region,
    "dimensions": 800,
    "format": "png",
})

annual_path = OUT_DIR / "mumbai_2020_mean_no2.png"
annual_path.write_bytes(requests.get(thumb_annual, timeout=180).content)
print("Saved", annual_path)


## 9. Monthly 2020 NO₂ (time slider + GIF)

Same cloud mask and same colour scale as the annual map.


In [ ]:
# 9. Build monthly mean NO2 ImageCollection for 2020
months = ee.List.sequence(1, 12)
month_names = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]


def monthly_no2(month):
    month = ee.Number(month)
    start = ee.Date.fromYMD(2020, month, 1)
    end = start.advance(1, "month")

    image = (
        no2_2020
        .filterDate(start, end)
        .map(mask_clouds)
        .select(NO2_BAND)
        .mean()
        .clip(mumbai_ee.geometry())
    )

    return image.set({
        "month": month,
        "system:time_start": start.millis(),
        "system:index": month.format("%02d"),
    })


monthly_collection = ee.ImageCollection.fromImages(months.map(monthly_no2))
print("Monthly images:", monthly_collection.size().getInfo())

# City-wide monthly means (for the summary table)
monthly_stats = []
for m in range(1, 13):
    start = ee.Date.fromYMD(2020, m, 1)
    end = start.advance(1, "month")
    month_img = (
        no2_2020
        .filterDate(start, end)
        .map(mask_clouds)
        .select(NO2_BAND)
        .mean()
        .clip(mumbai_ee.geometry())
    )
    mean_val = month_img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=mumbai_ee.geometry(),
        scale=1000,
        maxPixels=1e13,
        bestEffort=True,
    ).get(NO2_BAND)
    monthly_stats.append({
        "month": m,
        "label": month_names[m - 1],
        "mean_no2_mol_m2": mean_val.getInfo(),
    })

monthly_df = pd.DataFrame(monthly_stats)
print(monthly_df.to_string(index=False))


In [ ]:
# 10. Interactive time slider (dynamic output in the notebook)
Map_monthly = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map_monthly.add_time_slider(
    monthly_collection,
    vis_params=no2_vis,
    time_interval=1,
    bands=[NO2_BAND],
    date_format="YYYY-MM",
)
Map_monthly.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map_monthly


In [ ]:
# 11. Save monthly PNG frames + animated GIF + HTML time slider for the webpage
frames_dir = OUT_DIR / "monthly_frames"
frames_dir.mkdir(exist_ok=True)
frame_paths = []

for m in range(1, 13):
    start = ee.Date.fromYMD(2020, m, 1)
    end = start.advance(1, "month")
    month_img = (
        no2_2020
        .filterDate(start, end)
        .map(mask_clouds)
        .select(NO2_BAND)
        .mean()
        .clip(mumbai_ee.geometry())
    )
    url = month_img.getThumbURL({
        "min": no2_vis["min"],
        "max": no2_vis["max"],
        "palette": no2_vis["palette"],
        "region": region,
        "dimensions": 700,
        "format": "png",
    })
    raw_path = frames_dir / f"no2_2020_{m:02d}.png"
    raw_path.write_bytes(requests.get(url, timeout=180).content)

    im = Image.open(raw_path).convert("RGBA")
    bar = Image.new("RGBA", (im.width, 36), (11, 61, 74, 230))
    labeled = Image.new("RGBA", (im.width, im.height + 36), (255, 255, 255, 255))
    labeled.paste(bar, (0, 0))
    labeled.paste(im, (0, 36))
    draw = ImageDraw.Draw(labeled)
    draw.text(
        (12, 8),
        f"Greater Mumbai — {month_names[m - 1]} 2020 tropospheric NO2",
        fill=(255, 255, 255),
    )
    labeled_path = frames_dir / f"labeled_{m:02d}.png"
    labeled.convert("RGB").save(labeled_path)
    frame_paths.append(labeled_path)
    print("Saved frame", labeled_path.name)

gif_path = OUT_DIR / "mumbai_2020_monthly_no2.gif"
frames = [Image.open(p).convert("RGB") for p in frame_paths]
frames[0].save(
    gif_path,
    save_all=True,
    append_images=frames[1:],
    duration=700,
    loop=0,
)
print("Saved GIF", gif_path)

slider_html = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Mumbai 2020 monthly NO2 time slider</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 16px; background: #f4f7f8; color: #10262f; }
    img { max-width: 100%; border: 1px solid #c9d6dc; border-radius: 8px; background: white; }
    input[type=range] { width: min(640px, 100%); }
  </style>
</head>
<body>
  <h2>Greater Mumbai — monthly mean tropospheric NO2 (2020)</h2>
  <p>Drag the slider to change month. Colour scale is fixed across months.</p>
  <p><b>Month: <span id="label">Jan 2020</span></b></p>
  <input id="slider" type="range" min="1" max="12" value="1" step="1">
  <div style="margin-top:12px"><img id="frame" alt="Monthly NO2 map"></div>
  <script>
    const names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"];
    const slider = document.getElementById("slider");
    const frame = document.getElementById("frame");
    const label = document.getElementById("label");
    function update() {
      const m = String(slider.value).padStart(2, "0");
      frame.src = "monthly_frames/no2_2020_" + m + ".png";
      label.textContent = names[Number(slider.value) - 1] + " 2020";
    }
    slider.addEventListener("input", update);
    update();
  </script>
</body>
</html>
'''
slider_path = OUT_DIR / "mumbai_2020_no2_timeslider.html"
slider_path.write_text(slider_html, encoding="utf-8")
print("Saved", slider_path)


## 12. Highest-NO₂ quartile (city P75 rule)

**Rule:** pixels at or above the city-wide **75th percentile** of 2020 mean tropospheric NO₂  
are treated as the **highest-NO₂ quartile** (“relatively high” values).


In [ ]:
# 12. 75th percentile threshold and high-NO2 mask
no2_threshold = annual_no2.reduceRegion(
    reducer=ee.Reducer.percentile([75]),
    geometry=mumbai_ee.geometry(),
    scale=1000,
    maxPixels=1e13,
    bestEffort=True,
)

threshold = ee.Number(no2_threshold.get(NO2_BAND))
print("NO2 75th percentile threshold (mol/m2):", threshold.getInfo())

high_no2 = annual_no2.gte(threshold).selfMask()

Map_high = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map_high.addLayer(annual_no2, no2_vis, "Annual Mean NO2 - 2020")
Map_high.addLayer(high_no2, {"palette": ["#6a00a8"]}, "Highest NO2 Quartile (P75+)")
Map_high.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map_high.add_colorbar(no2_vis, label="Tropospheric NO2 (mol/m2)")
Map_high


## 13. WorldPop residents aged 65+ (India, 2020)

Sum male and female age bands from 65 years upward.  
This is gridded residential population — **not** personal exposure.


In [ ]:
# 13. Load WorldPop age-sex for India, 2020 and build population aged 65+
worldpop_age = ee.ImageCollection("WorldPop/GP/100m/pop_age_sex_cons_unadj")

india_pop = (
    worldpop_age
    .filter(ee.Filter.eq("country", "IND"))
    .filter(ee.Filter.eq("year", 2020))
)
print("India 2020 images:", india_pop.size().getInfo())

pop_image = ee.Image(india_pop.first())
print("Image properties:", pop_image.toDictionary().getInfo())

band_names = pop_image.bandNames().getInfo()
age65_bands = [
    b for b in band_names
    if "_" in b
    and b.split("_")[0] in ("M", "F", "m", "f")
    and b.split("_")[1].isdigit()
    and int(b.split("_")[1]) >= 65
]
print("65+ bands:", age65_bands)

pop65 = (
    pop_image
    .select(age65_bands)
    .reduce(ee.Reducer.sum())
    .rename("population_65_plus")
    .clip(mumbai_ee.geometry())
)

print(
    "65+ min/max:",
    pop65.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=mumbai_ee.geometry(),
        scale=100,
        maxPixels=1e13,
        bestEffort=True,
    ).getInfo(),
)

Map_65 = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map_65.addLayer(
    pop65,
    {"min": 0, "max": 20, "palette": ["yellow", "orange", "red", "darkred"]},
    "Population aged 65+",
)
Map_65.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map_65


## 14. Exposure: residents aged 65+ in the highest-NO₂ quartile

Report as **population residing in areas with relatively high satellite-observed NO₂**  
(tropospheric column). Do **not** treat this as ground-level personal exposure.


In [ ]:
# 14. 65+ population inside highest-NO2 quartile + summary statistics
exposed_65 = pop65.updateMask(high_no2)

exposed_total = exposed_65.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=mumbai_ee.geometry(),
    scale=100,
    maxPixels=1e13,
    bestEffort=True,
)
total_65 = pop65.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=mumbai_ee.geometry(),
    scale=100,
    maxPixels=1e13,
    bestEffort=True,
)
city_mean_no2 = annual_no2.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=mumbai_ee.geometry(),
    scale=1000,
    maxPixels=1e13,
    bestEffort=True,
).get(NO2_BAND)

exposed_value = exposed_total.getInfo()
total_value = total_65.getInfo()

exposed_population = float(exposed_value["population_65_plus"] or 0)
total_population = float(total_value["population_65_plus"] or 0)
exposure_share = (exposed_population / total_population * 100) if total_population else float("nan")
p75_value = threshold.getInfo()
mean_no2_value = city_mean_no2.getInfo()

print("City-wide mean NO2 (mol/m2):", mean_no2_value)
print("NO2 P75 threshold (mol/m2):", p75_value)
print("Exposed 65+ population:", round(exposed_population))
print("Total 65+ population:", round(total_population))
print("Share of 65+ in highest-NO2 areas:", round(exposure_share, 2), "%")

summary = {
    "city": "Greater Mumbai",
    "year": 2020,
    "no2_city_mean_mol_m2": mean_no2_value,
    "no2_p75_mol_m2": p75_value,
    "pop65_total": total_population,
    "pop65_in_highest_no2_quartile": exposed_population,
    "share_pop65_in_highest_no2_quartile_percent": exposure_share,
    "high_no2_rule": "Pixels >= city-wide 75th percentile of 2020 mean tropospheric NO2",
    "boundary_source": "dataset/Mumbai_boundary.geojson (GREATER MUMBAI ULB; SBM / ramSeraph urban release)",
    "monthly_mean_no2": monthly_stats,
}

summary_path = OUT_DIR / "mumbai_2020_no2_exposure_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved", summary_path)

summary_table = pd.DataFrame([
    {"Metric": "City-wide mean tropospheric NO2 (mol/m2)", "Value": mean_no2_value},
    {"Metric": "Highest-quartile threshold (P75, mol/m2)", "Value": p75_value},
    {"Metric": "Residents aged 65+ (city total)", "Value": round(total_population)},
    {"Metric": "Residents aged 65+ in highest-NO2 quartile", "Value": round(exposed_population)},
    {"Metric": "Share of 65+ in highest-NO2 quartile (%)", "Value": round(exposure_share, 2)},
])
table_path = OUT_DIR / "mumbai_2020_no2_exposure_table.csv"
summary_table.to_csv(table_path, index=False)
print(summary_table.to_string(index=False))
print("Saved", table_path)


In [ ]:
# 15. Exposure map (annual NO2 + highest quartile + 65+ in high-NO2 zone)
Map_exposure = geemap.Map(center=[19.0760, 72.8777], zoom=10)
Map_exposure.addLayer(annual_no2, no2_vis, "Annual Mean NO2 - 2020")
Map_exposure.addLayer(
    high_no2,
    {"palette": ["#6a00a8"]},
    "Highest NO2 Quartile (P75+)",
    True,
    0.45,
)
Map_exposure.addLayer(
    exposed_65,
    {"min": 0, "max": 20, "palette": ["yellow", "orange", "red", "darkred"]},
    "65+ population in highest-NO2 areas",
)
Map_exposure.addLayer(mumbai_ee, {"color": "black"}, "Greater Mumbai Boundary")
Map_exposure.add_colorbar(no2_vis, label="Tropospheric NO2 (mol/m2)")
Map_exposure


In [ ]:
# 16. Export exposure thumbnail for the webpage
exposure_rgb = annual_no2.visualize(**no2_vis)
high_vis = high_no2.visualize(min=0, max=1, palette=["#6a00a8"])
exposure_map = ee.ImageCollection([exposure_rgb, high_vis]).mosaic().clip(mumbai_ee.geometry())

thumb_exp = exposure_map.getThumbURL({
    "region": region,
    "dimensions": 800,
    "format": "png",
})
exposure_path = OUT_DIR / "mumbai_2020_no2_65plus_exposure.png"
exposure_path.write_bytes(requests.get(thumb_exp, timeout=180).content)
print("Saved", exposure_path)

print("\nWebpage outputs:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p.relative_to(OUT_DIR), f"({p.stat().st_size} bytes)")


## 17. Discussion

### Spatial pattern
Higher 2020 mean tropospheric NO₂ inside Greater Mumbai tends to concentrate over denser built-up / traffic-intensive parts of the ULB. The P75 mask therefore highlights a relatively high core on the coarse Sentinel-5P grid (~3.5–7 km), not fine street-level hotspots.

### Temporal pattern
The monthly time slider and GIF keep one fixed colour scale, so month-to-month differences reflect changes in the cloud-masked mean column. Monsoon months can look quieter partly because cloud filtering removes many scenes; winter months often show stronger NO₂ signals in Indian megacities. Read monthly change together with data availability, not only as emissions change.

### Exposure reading
The key statistic is the share of WorldPop residents aged 65+ who live in pixels at or above the city-wide 75th percentile of 2020 mean tropospheric NO₂. That is a transparent, city-specific definition of “relatively high” **satellite-observed** NO₂. It is **not** a health standard and must not be described as individual or ground-level personal exposure.

### Limitations
- S5P NO₂ is a tropospheric **column** density, not roadside concentration  
- Coarse spatial resolution smooths neighbourhood differences  
- Cloud masking / monsoon clouds reduce valid observations in some months  
- WorldPop 2020 age–sex surfaces are modelled residential estimates (~100 m)  
- ULB boundary extents can differ slightly from other official GIS layers  

### Webpage
Maps, GIF, time slider, table, and this discussion are hosted at `web/q2/index.html` (live `/q2/` after deploy).
